# HKDN - Train YOLO11 chuyên biệt Dẫn đường Người Khiếm Thị (Google Colab T4 GPU)

Notebook này tự động:
1. Tải bộ dữ liệu thực tế cho người khiếm thị từ HuggingFace (`Abtinzandi/Obstacle-Detection-Dataset-YOLO`, 24.326 ảnh, có `Stairs`, `Manhole`, `Electrical Pole`, `Guard rail`, `Person`, `Chair`, `Bench`, `Motorcycle`...)
2. Tùy chọn gộp thêm ảnh tự chụp / `ImagesTest/` của bạn (nếu clone repo về Colab)
3. Ánh xạ về đúng **15 lớp chuẩn** của hệ thống HKDN và train `yolo11s.pt` trên GPU T4 (~20-30 phút)
4. Tự động đóng gói tải về file **`guidance_model.pt`** — bạn chỉ cần bỏ file này vào thư mục gốc dự án là `yolo_service.py` tự động nạp.

In [ ]:
!pip -q install ultralytics huggingface_hub pyyaml
import torch
print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Clone nhanh repo HKDN để dùng sẵn script chuẩn hóa nhãn `prepare_and_train_yolo.py`
import os
if not os.path.exists('AI-Powered-Visual-Guidance-System'):
    !git clone -b feature/Vuong_demo-20/11 https://github.com/VanQuyVuong/AI-Powered-Visual-Guidance-System.git
%cd /content/AI-Powered-Visual-Guidance-System

In [ ]:
# Tải & gộp bộ dữ liệu 24.326 ảnh ROD-Dataset từ HuggingFace (đặt --max-samples 0 để tải FULL 24K ảnh, hoặc 4000 để test nhanh ~10 phút)
!python backend/training/prepare_and_train_yolo.py --mode download-hf --max-samples 4000
# Gộp thêm các ảnh trong thư mục ImagesTest (nếu bạn có upload lên Colab)
!python backend/training/prepare_and_train_yolo.py --mode auto-label

In [ ]:
# Train YOLO11s trên GPU T4 (50 epochs, imgsz=480)
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
results = model.train(
    data='backend/training/custom_dataset/data.yaml',
    epochs=50,
    imgsz=480,
    batch=32,
    project='runs',
    name='hkdn_guidance_yolo11s',
)

In [ ]:
# Xuất & tải thẳng file `guidance_model.pt` về máy tính
import shutil
from pathlib import Path
from google.colab import files

best_pt = Path('runs/hkdn_guidance_yolo11s/weights/best.pt')
out_pt = Path('guidance_model.pt')
shutil.copy2(best_pt, out_pt)
print('Saved:', out_pt.resolve(), out_pt.stat().st_size, 'bytes')
files.download(str(out_pt))